# VidSig / Video Signature: four inpainting methods

Repository: [https://github.com/hardenyu21/Video-Signature](https://github.com/hardenyu21/Video-Signature)

Paper: [https://arxiv.org/abs/2506.00652](https://arxiv.org/abs/2506.00652)

Inspected commit: `9d219c47670c30fdaaccd17b24e7783c976a8c66`

Watermark stage: `in_generation`
Domain: video

This method is coupled to generation, so the notebook uses only author-provided or author-generated native data. It does not fake a MOSE embedding.

ProPainter, DiffuEraser, EffectErase, and E2FGVI-HQ receive the same authentic watermarked frames and binary mask. One official DiffuEraser execution is used: its internal ProPainter prior is preserved as the ProPainter result, and its final refinement is the DiffuEraser result. Inputs shorter than 22 frames are padded by repeating only the last frame/mask to 24 for that pipeline, then both reported outputs are trimmed to the authentic length. EffectErase applies its released Wan 4n+1 temporal adapter; E2FGVI-HQ uses the authors' official arbitrary-resolution test entry point.

The native sample is generated from the first prompt in the authors' `prompt.txt`, their ModelScope fine-tuned VAE, released 48-bit TorchScript decoder, published key, and seed 42. No object masks are released, so the notebook records a deterministic center-region mask.


## Authenticity contract

- Watermark preparation: Download the authors' fine-tuned checkpoint, generate the clean and watermarked videos with src/generate_ms.py, and build a mask aligned to the watermarked result.
- Checkpoint status: REQUIRES the authors' fine-tuned checkpoint distributed through their Google Drive link.
- Protocol note: VidSig modifies the video generator.  Its watermark cannot be added faithfully to an existing MOSE video after generation.
- No result cell fabricates metrics. Native extraction must finish and write its JSON before values are displayed.
- A method without a released spatial detector reports bit recovery only; it is not assigned a proxy localizer.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/remaining_watermark_methods.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"

def run(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", shlex.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def require(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path

def require_commit(repository, expected):
    actual = subprocess.check_output(["git", "-C", str(repository), "rev-parse", "HEAD"], text=True).strip()
    if actual != expected:
        raise RuntimeError(f"Expected {expected} in {repository}, found {actual}")

def show_results(run_root):
    run_root = Path(run_root)
    for name in ("watermarked.mp4", "aligned-mask.mp4"):
        path = run_root / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    attack = run_root / "attack"
    for name in (
        "input-watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4",
        "effecterase.mp4", "e2fgvi.mp4",
    ):
        path = attack / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    result_json = {
        *run_root.rglob("summary.json"),
        *(attack / name for name in ("sample.json", "removal-pair.json", "effecterase.json", "e2fgvi.json")),
    }
    for path in sorted(path for path in result_json if path.is_file()):
        print(path.relative_to(run_root))
        display(JSON(json.loads(path.read_text())))

print("Repository:", ROOT)


## Canonical one-clip comparison: all four inpainters

This is the canonical comparison for the minimum one-clip requirement. All four removers use the same frozen watermarked input and aligned binary mask. Every remover is presented in the same order, with the same output-level fields and the same video/JSON evidence. Method-specific resizing, padding, and mask dilation remain recorded in JSON rather than being silently normalized.

| Inpainting method | Status | Runtime (s) | Frames | Dimensions | FPS | Bytes | Evidence |
|---|---:|---:|---:|---:|---:|---:|---|
| ProPainter | complete | not recorded | 16 | 512 × 512 | 10.000 | 216,217 | [video](../../runs/remaining-watermarks/vidsig/red-panda-seed42/attack/propainter.mp4) · [JSON](../../runs/remaining-watermarks/vidsig/red-panda-seed42/attack/propainter.json) |
| DiffuEraser | complete | not recorded | 16 | 512 × 512 | 10.000 | 516,470 | [video](../../runs/remaining-watermarks/vidsig/red-panda-seed42/attack/diffueraser.mp4) · [JSON](../../runs/remaining-watermarks/vidsig/red-panda-seed42/attack/diffueraser.json) |
| EffectErase | complete | 49.137 | 16 | 512 × 512 | 10.000 | 453,069 | [video](../../runs/remaining-watermarks/vidsig/red-panda-seed42/attack/effecterase.mp4) · [JSON](../../runs/remaining-watermarks/vidsig/red-panda-seed42/attack/effecterase.json) |
| E2FGVI-HQ | complete | 7.886 | 16 | 512 × 512 | 10.000 | 600,394 | [video](../../runs/remaining-watermarks/vidsig/red-panda-seed42/attack/e2fgvi.mp4) · [JSON](../../runs/remaining-watermarks/vidsig/red-panda-seed42/attack/e2fgvi.json) |

Watermark method: **Vidsig**. Runtime is shown only when the original run record captured it; `not recorded` is not replaced with an estimate. Watermark-specific localization or recovery metrics are downstream evaluations and are kept separate from this equal inpainting comparison.


In [ ]:
from pathlib import Path
import json
from IPython.display import JSON, Markdown, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/finalize_four_inpainter_notebooks.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
ATTACK = ROOT / 'runs/remaining-watermarks/vidsig/red-panda-seed42/attack'
for name in ("input-watermarked.mp4", "watermarked.mp4", "gt-mask.mp4"):
    path = ATTACK / name
    if path.is_file():
        print(name)
        display(Video(str(path), embed=True, html_attributes="controls muted loop"))

REMOVERS = (
    ("ProPainter", "propainter"),
    ("DiffuEraser", "diffueraser"),
    ("EffectErase", "effecterase"),
    ("E2FGVI-HQ", "e2fgvi"),
)
for label, stem in REMOVERS:
    display(Markdown(f"### {label}"))
    video = ATTACK / f"{stem}.mp4"
    metadata = ATTACK / f"{stem}.json"
    if not video.is_file() or not metadata.is_file():
        raise FileNotFoundError(f"Missing {label} evidence in {ATTACK}")
    display(Video(str(video), embed=True, html_attributes="controls muted loop"))
    display(JSON(json.loads(metadata.read_text())))

for name in ("sample.json", "removal-pair.json"):
    metadata = ATTACK / name
    if metadata.is_file():
        print(name)
        display(JSON(json.loads(metadata.read_text())))


In [ ]:
VIDSIG = Path("/path/to/Video-Signature")
VIDSIG_PYTHON = VIDSIG / ".venv/bin/python"
VAE_CHECKPOINT = VIDSIG / "ckpts/vae/modelscope/checkpoint.pth"
MESSAGE_DECODER = VIDSIG / "ckpts/msg_decoder/dec_48b_whit.torchscript.pt"
DIFFUERASER = Path("/path/to/DiffuEraser")
DIFFUERASER_PYTHON = DIFFUERASER / ".venv/bin/python"
GPU = "0"
RUN_ROOT = ROOT / "runs/remaining-watermarks/vidsig/red-panda-seed42"
NATIVE = RUN_ROOT / "native-generation"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WATERMARKED_VIDEO = NATIVE / "watermarked.mp4"
ALIGNED_MASK = RUN_ROOT / "aligned-mask.mp4"


In [ ]:
require_commit(VIDSIG, "9d219c47670c30fdaaccd17b24e7783c976a8c66")
require(VAE_CHECKPOINT, "authors' fine-tuned ModelScope VAE")
require(MESSAGE_DECODER, "authors' 48-bit TorchScript decoder")


In [ ]:
if not WATERMARKED_VIDEO.is_file():
    run([
        VIDSIG_PYTHON, SCRIPTS / "run_vidsig_native_sample.py", "--source", VIDSIG,
        "--vae-checkpoint", VAE_CHECKPOINT, "--message-decoder", MESSAGE_DECODER,
        "--output", NATIVE, "--prompt-file", VIDSIG / "prompt.txt",
        "--prompt-index", "0", "--seed", "42",
    ])
run([
    DIFFUERASER_PYTHON, SCRIPTS / "make_native_removal_mask.py", "center-box",
    "--video", WATERMARKED_VIDEO, "--output-video", ALIGNED_MASK,
    "--output-frames", RUN_ROOT / "aligned-mask-frames",
])


## Run all four inpainting methods on the frozen input

The preparation cell freezes hashes, frame count, dimensions, FPS, mask foreground counts, provenance, and any short-video padding. Every remover below consumes that frozen input and mask.


In [ ]:
ATTACK_RUN = RUN_ROOT / "attack"
if not (ATTACK_RUN / "sample.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "prepare_watermark_removal_sample.py", "prepare",
        "--source-video", WATERMARKED_VIDEO, "--source-mask", ALIGNED_MASK,
        "--output", ATTACK_RUN, "--watermark-method", "VidSig",
        "--sample-id", "red-panda-prompt-0-seed-42", "--sample-source", "Video Signature authors' prompt, key, checkpoints, and ModelScope pipeline",
        "--source-kind", "authors_generated",
    ])
else:
    print("Prepared sample already exists:", ATTACK_RUN / "sample.json")


In [ ]:
if not (ATTACK_RUN / "removal-pair.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "run_watermark_removal_pair.py",
        "--run", ATTACK_RUN, "--diffueraser", DIFFUERASER,
        "--diffueraser-python", DIFFUERASER_PYTHON,
        "--device", GPU, "--seed", "2025",
    ])
else:
    print("Both removal outputs already exist:", ATTACK_RUN / "removal-pair.json")


In [ ]:
EFFECTERASE = Path(os.environ.get("EFFECTERASE", "/mnt/data_fast/smayan/research/EffectErase"))
EFFECTERASE_PYTHON = Path(os.environ.get("EFFECTERASE_PYTHON", str(EFFECTERASE / ".venv/bin/python")))
E2FGVI = Path(os.environ.get("E2FGVI", "/mnt/data_fast/smayan/research/E2FGVI"))
E2FGVI_CHECKPOINT = Path(os.environ.get(
    "E2FGVI_CHECKPOINT", str(E2FGVI / "release_model/E2FGVI-HQ-CVPR22.pth")
))
E2FGVI_PYTHON = Path(os.environ.get("E2FGVI_PYTHON", str(DIFFUERASER_PYTHON)))

if not (ATTACK_RUN / "effecterase.json").is_file():
    effect_env = os.environ.copy(); effect_env["CUDA_VISIBLE_DEVICES"] = GPU
    run([
        EFFECTERASE_PYTHON, SCRIPTS / "run_prepared_effecterase.py",
        "--source", EFFECTERASE, "--run", ATTACK_RUN, "--device", GPU,
    ], env=effect_env)
else:
    print("EffectErase output already exists:", ATTACK_RUN / "effecterase.json")

if not (ATTACK_RUN / "e2fgvi.json").is_file():
    run([
        E2FGVI_PYTHON, SCRIPTS / "run_prepared_e2fgvi.py",
        "--source", E2FGVI, "--checkpoint", E2FGVI_CHECKPOINT,
        "--run", ATTACK_RUN, "--python", E2FGVI_PYTHON, "--device", GPU,
    ])
else:
    print("E2FGVI-HQ output already exists:", ATTACK_RUN / "e2fgvi.json")


In [ ]:
run([
    VIDSIG_PYTHON, SCRIPTS / "run_vidsig_detection.py",
    "--source", VIDSIG, "--decoder", MESSAGE_DECODER,
    "--output", RUN_ROOT / "detection",
    "--case", f"original={ATTACK_RUN / 'input-watermarked.mp4'}",
    "--case", f"propainter={ATTACK_RUN / 'propainter.mp4'}",
    "--case", f"diffueraser={ATTACK_RUN / 'diffueraser.mp4'}",
    "--case", f"effecterase={ATTACK_RUN / 'effecterase.mp4'}",
    "--case", f"e2fgvi={ATTACK_RUN / 'e2fgvi.mp4'}",
])


In [ ]:
show_results(RUN_ROOT)
